# Lorenz, fitted equations: entropy transfers (MF)

Same as `lorenz-true.ipynb`, but $F_2$, $F_3$ are fitted from data (Sec. 4.3). Own run: the fit divides by `dt`, so it must match the simulation.

## Parameters

Lorenz constants and scaling (Sec. 6); `dt` (Sec. 6.2, extra MF run).

In [1]:
import random, torch
torch.set_default_dtype(torch.float64)

sig, Ra, beta = 10.0, 27.0, 8 / 3
mu3, s1, s2, s3 = 22.61, 7.74, 8.76, 8.35
dt = 0.01

## Equations (Eq. 71a–c, noise off)

$$
\frac{dX_1}{dt} = \sigma\frac{s_2}{s_1}X_2 - \sigma X_1
$$

$$
\frac{dX_2}{dt} = \frac{s_1}{s_2}(R_a - \mu_3)X_1 - \frac{s_1 s_3}{s_2}X_1X_3 - X_2
$$

$$
\frac{dX_3}{dt} = \frac{s_1 s_2}{s_3}X_1X_2 - \beta X_3 - \beta\frac{\mu_3}{s_3}
$$

In [2]:
def F(x1, x2, x3):
    return [
        (sig * s2 / s1) * x2 - sig * x1,
        (s1 / s2) * (Ra - mu3) * x1 - (s1 * s3 / s2) * x1 * x3 - x2,
        (s1 * s2 / s3) * x1 * x2 - beta * x3 - beta * mu3 / s3,
    ]

## Simulate (App. B2, Eq. B.6–B.8)

$$
X^* = X + dt\,F(X), \qquad X^{**} = X + dt\,F(X^*), \qquad X_{\text{new}} = \tfrac12\,(X^* + X^{**})
$$

Drop the first 100 time units, keep 400 (Sec. 6.2). One run, random start (paper: 30 runs).

In [3]:
random.seed(0)
X = [random.gauss(0, 1) for _ in range(3)]
run = []
for t in range(int(500 / dt)):
    f = F(*X)
    Xs = [X[i] + dt * f[i] for i in range(3)]          # B.6
    fs = F(*Xs)
    Xss = [X[i] + dt * fs[i] for i in range(3)]        # B.7
    X = [0.5 * (Xs[i] + Xss[i]) for i in range(3)]     # B.8
    if t >= int(100 / dt):
        run.append(X)

torch.save(run, "lorenz_run_fitted.pt")

In [4]:
X1 = [p[0] for p in run]
X2 = [p[1] for p in run]
X3 = [p[2] for p in run]
N = len(run)
print(N, "points")

40000 points


## Fit the equations (Eq. 47a, App. B1)

$$
\frac{dX_i}{dt} \approx \theta_1 + \theta_{X_1}X_1 + \theta_{X_2}X_2 + \theta_{X_3}X_3 + \theta_{X_1^2}X_1^2 + \theta_{X_1X_2}X_1X_2 + \theta_{X_1X_3}X_1X_3 + \theta_{X_2^2}X_2^2 + \theta_{X_2X_3}X_2X_3 + \theta_{X_3^2}X_3^2
$$

Rate $= (X_i[n+1] - X_i[n])/dt$; least squares (Eq. B.1).

True values (Eq. 71b, 71c), all other terms 0:

| term | $X_2$ eq | $X_3$ eq |
|---|---|---|
| constant | 0 | −7.218 |
| $X_1$ | 3.879 | 0 |
| $X_2$ | −1 | 0 |
| $X_3$ | 0 | −2.667 |
| $X_1X_2$ | 0 | 8.121 |
| $X_1X_3$ | −7.378 | 0 |

In [5]:
names = ["1", "X1", "X2", "X3", "X1^2", "X1X2", "X1X3", "X2^2", "X2X3", "X3^2"]

def terms(a, b, c):
    return [1, a, b, c, a * a, a * b, a * c, b * b, b * c, c * c]

def fit_equation(Xi):
    A = [terms(X1[n], X2[n], X3[n]) for n in range(N - 1)]
    rate = [[(Xi[n + 1] - Xi[n]) / dt] for n in range(N - 1)]
    th = torch.linalg.lstsq(torch.tensor(A), torch.tensor(rate)).solution[:, 0].tolist()
    return dict(zip(names, th))

th = fit_equation(X2)
th3 = fit_equation(X3)
for nm in names:
    print(f"{nm:5s}  X2 eq {th[nm]:7.3f}   X3 eq {th3[nm]:7.3f}")

1      X2 eq  -0.011   X3 eq  -7.065
X1     X2 eq   3.994   X3 eq  -0.016
X2     X2 eq  -1.451   X3 eq   0.010
X3     X2 eq  -0.004   X3 eq  -2.838
X1^2   X2 eq   0.002   X3 eq   0.190
X1X2   X2 eq  -0.002   X3 eq   6.678
X1X3   X2 eq  -6.865   X3 eq   0.006
X2^2   X2 eq  -0.001   X3 eq   1.095
X2X3   X2 eq  -0.750   X3 eq  -0.006
X3^2   X2 eq   0.002   X3 eq  -0.161


## Tool 1: `fit_curve(x, y)`

Degree-10 polynomial through the cloud: the curve $E(y \mid x)$ (Sec. 4.2; $N_d = 10$, Sec. 6.2).

Dummy: points on $1 + 2x + 3x^2$, degree 2 → `[1, 2, 3]`.

In [6]:
def fit_curve(x, y, deg=10):
    A = [[xn ** p for p in range(deg + 1)] for xn in x]
    Y = [[yn] for yn in y]
    return torch.linalg.lstsq(torch.tensor(A), torch.tensor(Y)).solution[:, 0].tolist()

In [7]:
xs = [0.0, 1.0, 2.0, 3.0]
print(fit_curve(xs, [1 + 2 * x + 3 * x ** 2 for x in xs], deg=2))

[0.9999999999999918, 2.00000000000001, 2.999999999999997]


## Tool 2: `value(c, x)`, `slope(c, x)`

Height and derivative of the curve at `x`.

Dummy: `c = [1, 2, 3]` at $x = 2$ → 17, 14.

In [8]:
def value(c, x):
    return sum(c[p] * x ** p for p in range(len(c)))

def slope(c, x):
    return sum(p * c[p] * x ** (p - 1) for p in range(1, len(c)))

In [9]:
print(value([1, 2, 3], 2), slope([1, 2, 3], 2))

17 14


## All $\to X_3$ (Eq. 11, Sec. 2.2)

$$
T(X_{\sim 3}\to X_3) = E\Big[\frac{d}{dX_3}E\big(F_{3,\sim 3}\mid X_3\big)\Big] - E\Big[\frac{\partial F_{3,\sim 3}}{\partial X_3}\Big]
$$

$F_{3,\sim 3}$ = terms of $F_3$ with $X_1$ or $X_2$:

$$
F_{3,\sim 3} = \theta_{X_1}X_1 + \theta_{X_2}X_2 + \theta_{X_1^2}X_1^2 + \theta_{X_1X_2}X_1X_2 + \theta_{X_1X_3}X_1X_3 + \theta_{X_2^2}X_2^2 + \theta_{X_2X_3}X_2X_3
$$

In [10]:
cross = ["X1", "X2", "X1^2", "X1X2", "X1X3", "X2^2", "X2X3"]

F3_cross = []
for n in range(N):
    t = dict(zip(names, terms(X1[n], X2[n], X3[n])))
    F3_cross.append(sum(th3[nm] * t[nm] for nm in cross))

First term: curve against $X_3$ → slope at each point → average.

In [11]:
c = fit_curve(X3, F3_cross)

first = 0.0
for n in range(N):
    first += slope(c, X3[n])
first = first / N
print("first term:", first)

first term: 2.4776494059096446


Second term: frozen slope $\theta_{X_1X_3}X_1 + \theta_{X_2X_3}X_2$, averaged.

True equation: 2.633.

In [12]:
second = 0.0
for n in range(N):
    second += th3["X1X3"] * X1[n] + th3["X2X3"] * X2[n]
second = second / N
print("second term:", second)
print("T(X1,X2 -> X3):", first - second, "  (true eq: 2.633)")

second term: -8.706145926356244e-06
T(X1,X2 -> X3): 2.477658112055571   (true eq: 2.633)


## One $\to X_2$ (Eq. 37a, Corollary 4, p. 14)

For a term $F_{i,j} = f_1(X_j)\,f_{2a}(X_i)\,f_{2b}(X_k)$:

$$
T(X_j \to X_i)_F = E_i\Big[f_{2a}(X_i)\, E_k(f_{2b} \mid X_i)\, \frac{d}{dX_i} E_j(f_1 \mid X_i)\Big]
$$

$f_1$: driver part. $f_{2a}$: target part. $f_{2b}$: outer part.

Only terms with $X_j$ count (Eq. 27). Apply to each term and add (p. 12, p. 14).

Paper's example (Eq. 38a, 39a): $dX_1/dt = \exp(X_1)\sin(X_2) + X_1X_3$, so $f_1 = \sin X_2$, $f_{2a} = \exp X_1$:

$$
T(X_2 \to X_1)_F = E_1\Big[\exp(X_1)\,\frac{dE_2(\sin X_2 \mid X_1)}{dX_1}\Big]
$$

## Tool 3: `term37a(theta, f1, f2a, f2b)` (Eq. 37a, 50a)

One term $\hat\theta\, f_1(X_j)\, f_{2a}(X_2)\, f_{2b}(X_k)$:

$$
\hat\theta\; E_2\Big[f_{2a}(X_2)\; E(f_{2b} \mid X_2)\; \frac{dE(f_1 \mid X_2)}{dX_2}\Big]
$$

$\hat\theta$ out front; $f_{2a}$ as is; $f_{2b}$ through its curve; $f_1$ through the slope of its curve. Missing factor = 1. Add the terms (Eq. 49a).

In [13]:
def term37a(theta, f1, f2a=None, f2b=None):
    d = fit_curve(X2, f1)
    if f2b is not None:
        b = fit_curve(X2, f2b)
    total = 0.0
    for n in range(N):
        a = f2a[n] if f2a is not None else 1.0
        e = value(b, X2[n]) if f2b is not None else 1.0
        total += a * e * slope(d, X2[n])
    return theta * total / N

### $X_3 \to X_2$

| term | $\hat\theta$ | $f_1$ | $f_{2a}$ | $f_{2b}$ |
|---|---|---|---|---|
| $\theta_{X_3}X_3$ | $\theta_{X_3}$ | $X_3$ | | |
| $\theta_{X_1X_3}X_1X_3$ | $\theta_{X_1X_3}$ | $X_3$ | | $X_1$ |
| $\theta_{X_2X_3}X_2X_3$ | $\theta_{X_2X_3}$ | $X_3$ | $X_2$ | |
| $\theta_{X_3^2}X_3^2$ | $\theta_{X_3^2}$ | $X_3^2$ | | |

Paper: −0.80 (Table 2, MF, dt 0.02).

In [14]:
T = (term37a(th["X3"], X3)
     + term37a(th["X1X3"], X3, f2b=X1)
     + term37a(th["X2X3"], X3, f2a=X2)
     + term37a(th["X3^2"], [x * x for x in X3]))
print("T(X3 -> X2):", T, "  (paper: -0.80)")

T(X3 -> X2): -0.735437847174678   (paper: -0.80)


### $X_1 \to X_2$

| term | $\hat\theta$ | $f_1$ | $f_{2a}$ | $f_{2b}$ |
|---|---|---|---|---|
| $\theta_{X_1}X_1$ | $\theta_{X_1}$ | $X_1$ | | |
| $\theta_{X_1X_2}X_1X_2$ | $\theta_{X_1X_2}$ | $X_1$ | $X_2$ | |
| $\theta_{X_1X_3}X_1X_3$ | $\theta_{X_1X_3}$ | $X_1$ | | $X_3$ |
| $\theta_{X_1^2}X_1^2$ | $\theta_{X_1^2}$ | $X_1^2$ | | |

Paper: 3.83 (Table 2, MF, dt 0.02).

In [15]:
T = (term37a(th["X1"], X1)
     + term37a(th["X1X2"], X1, f2a=X2)
     + term37a(th["X1X3"], X1, f2b=X3)
     + term37a(th["X1^2"], [x * x for x in X1]))
print("T(X1 -> X2):", T, "  (paper: 3.83)")

T(X1 -> X2): 3.717433879423986   (paper: 3.83)
